In [2]:
!module unload anaconda3/2024.06

In [2]:
!which python

/projects/frink/wang.xil/saelens_env/bin/python


In [3]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import argparse
from tqdm import tqdm
from datasets import load_dataset
from pathlib import Path
import json
import re

/projects/frink/wang.xil/saelens_env/lib/python3.10/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
/projects/frink/wang.xil/saelens_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Load OLMo 3 7B Think model
def load_model(model_name="allenai/OLMo-2-0325-32B-Instruct", device="cuda"):
    """
    Load OLMo model and tokenizer.
    
    Args:
        model_name: HuggingFace model name
        device: Device to load model on
    
    Returns:
        model, tokenizer
    """
    print(f"Loading model: {model_name}")
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,
        device_map="auto"
    )
    
    print(f"Model loaded on: {next(model.parameters()).device}")
    return model, tokenizer

model, tokenizer = load_model()

In [ ]:
# Parse functions for model output

def parse_chain_of_thought(model_output):
    """
    Extract the chain-of-thought reasoning before the </think> tag.
    
    Args:
        model_output: Raw model output string
    
    Returns:
        String containing the reasoning, or empty string if not found
    """
    if not model_output or not isinstance(model_output, str):
        return ''
    
    if '</think>' in model_output:
        parts = model_output.split('</think>', 1)
        return parts[0].strip()
    
    return ''


def parse_mcq_answer(model_output):
    """
    Parse the multiple choice answer (A/B/C/D/E) from model output.
    
    Args:
        model_output: Raw model output string
    
    Returns:
        'A', 'B', 'C', 'D', 'E', or 'Unparseable' if answer cannot be extracted
    """
    if not model_output or not isinstance(model_output, str):
        return 'Unparseable'
    
    # Remove chain of thought if present
    if '</think>' in model_output:
        model_output = model_output.split('</think>', 1)[1]
    
    # Look for "Answer: X" pattern
    match = re.search(r'Answer:\s*([A-Ea-e])', model_output, re.IGNORECASE)
    if match:
        return match.group(1).upper()
    
    # Look for "The answer is X" pattern
    match = re.search(r'[Tt]he answer is\s*([A-Ea-e])', model_output)
    if match:
        return match.group(1).upper()
    
    # Look for standalone letter at the start or after newline
    match = re.search(r'(?:^|\n)\s*([A-Ea-e])(?:\.|:|\s|$)', model_output)
    if match:
        return match.group(1).upper()
    
    # Fallback: return the last mentioned letter A-E
    letters = re.findall(r'\b([A-Ea-e])\b', model_output)
    if letters:
        return letters[-1].upper()
    
    return 'Unparseable'

In [ ]:
# Load medQA dataset
def load_medqa(filepath):
    """Load medQA from jsonl file."""
    data = []
    with open(filepath, 'r') as f:
        for line in f:
            data.append(json.loads(line))
    return data

# Load test set
medqa_test = load_medqa('/projects/frink/wang.xil/spurious_inject/data_clean/questions/US/test.jsonl')
print(f"Loaded {len(medqa_test)} test questions")

# Preview a sample
print("\nSample question:")
print(json.dumps(medqa_test[0], indent=2))

In [ ]:
# Create prompt for medQA question
def create_medqa_prompt(item):
    """
    Create a prompt for a medQA multiple choice question.
    
    Args:
        item: Dictionary with 'question', 'options', 'answer_idx'
    
    Returns:
        Formatted prompt string
    """
    question = item['question']
    options = item['options']
    
    # Format options
    options_text = "\n".join([f"{k}. {v}" for k, v in sorted(options.items())])
    
    prompt = f"""Answer the following medical question by selecting the correct option (A, B, C, D, or E).

Question: {question}

Options:
{options_text}

Provide your answer in the format "Answer: X" where X is the letter of your choice."""
    
    return prompt

# Test prompt creation
sample_prompt = create_medqa_prompt(medqa_test[0])
print(sample_prompt)

In [ ]:
# Run inference on medQA
def run_medqa_inference(model, tokenizer, data, max_samples=None, max_new_tokens=1024, 
                        temperature=0.6, top_p=0.95):
    """
    Run inference on medQA dataset.
    
    Args:
        model: Loaded model
        tokenizer: Loaded tokenizer
        data: List of medQA items
        max_samples: Maximum number of samples to process (None for all)
        max_new_tokens: Maximum tokens to generate
        temperature: Sampling temperature
        top_p: Top-p sampling parameter
    
    Returns:
        List of result dictionaries
    """
    if max_samples:
        data = data[:max_samples]
    
    results = []
    
    for idx, item in tqdm(enumerate(data), total=len(data), desc="Running inference"):
        prompt = create_medqa_prompt(item)
        message = [{"role": "user", "content": prompt}]
        
        inputs = tokenizer.apply_chat_template(
            message,
            add_generation_prompt=True,
            return_tensors='pt',
            return_dict=True
        ).to(model.device)
        
        input_length = inputs['input_ids'].shape[1]
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=temperature if temperature > 0 else 0.01,
                top_p=top_p,
                do_sample=temperature > 0,
                pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id
            )
        
        answer_only = tokenizer.decode(
            outputs[0][input_length:],
            skip_special_tokens=True
        ).strip()
        
        chain_of_thought = parse_chain_of_thought(answer_only)
        parsed_answer = parse_mcq_answer(answer_only)
        
        result = {
            'index': idx,
            'question': item['question'],
            'options': item['options'],
            'ground_truth': item['answer_idx'],
            'meta_info': item.get('meta_info', ''),
            'model_output': answer_only,
            'chain_of_thought': chain_of_thought,
            'parsed_answer': parsed_answer,
            'correct': parsed_answer == item['answer_idx']
        }
        
        results.append(result)
    
    return results

In [ ]:
# Run inference (adjust max_samples as needed)
results = run_medqa_inference(model, tokenizer, medqa_test, max_samples=100)

In [ ]:
# Compute and display statistics
def compute_medqa_statistics(results):
    """Compute accuracy and other statistics from results."""
    df = pd.DataFrame(results)
    
    total = len(df)
    correct = df['correct'].sum()
    accuracy = correct / total * 100
    
    unparseable = (df['parsed_answer'] == 'Unparseable').sum()
    
    print("="*60)
    print("MEDQA RESULTS")
    print("="*60)
    print(f"\nTotal questions: {total}")
    print(f"Correct answers: {correct}")
    print(f"Accuracy: {accuracy:.2f}%")
    print(f"Unparseable: {unparseable} ({unparseable/total*100:.2f}%)")
    
    # Breakdown by meta_info (e.g., step1, step2, step3)
    if 'meta_info' in df.columns and df['meta_info'].notna().any():
        print("\n--- Accuracy by Category ---")
        for cat in sorted(df['meta_info'].unique()):
            cat_df = df[df['meta_info'] == cat]
            cat_acc = cat_df['correct'].mean() * 100
            print(f"  {cat}: {cat_acc:.2f}% ({cat_df['correct'].sum()}/{len(cat_df)})")
    
    # Answer distribution
    print("\n--- Answer Distribution ---")
    print("Predicted:")
    print(df['parsed_answer'].value_counts())
    print("\nGround Truth:")
    print(df['ground_truth'].value_counts())
    
    print("="*60)
    
    return df

results_df = compute_medqa_statistics(results)

In [ ]:
# Examine individual results
def examine_result(results, idx):
    """Display a single result in detail."""
    r = results[idx]
    print(f"Question {idx}:")
    print("-"*50)
    print(f"Q: {r['question'][:200]}...")
    print(f"\nOptions: {r['options']}")
    print(f"\nGround Truth: {r['ground_truth']}")
    print(f"Parsed Answer: {r['parsed_answer']}")
    print(f"Correct: {r['correct']}")
    print(f"\n--- Chain of Thought ---")
    print(r['chain_of_thought'][:500] if r['chain_of_thought'] else "(none)")
    print(f"\n--- Full Model Output ---")
    print(r['model_output'][:1000])

# Example: examine first result
examine_result(results, 0)

In [ ]:
# Save results to CSV
output_path = '/projects/frink/wang.xil/spurious_inject/medqa_olmo_results.csv'
results_df.to_csv(output_path, index=False)
print(f"Results saved to: {output_path}")